In [1]:
# ============================================================
# SEGMENTATION CLASS WEIGHT COMPUTATION
# Pixel-level imbalance from FULL TRAIN SPLIT
# ============================================================

from pathlib import Path
from collections import Counter
import numpy as np
from PIL import Image
from tqdm import tqdm
import torch

In [ ]:


# ============================================================
# PATHS
# ============================================================

# CHANGE THIS IF NEEDED
PROJECT_ROOT = Path.cwd()

# Example:
# dataset/
#   gtFine/
#       train/
#       val/

MASKS_DIR = PROJECT_ROOT / "data" / "gtFine" / "train"

# ============================================================
# CONFIG
# ============================================================

NUM_CLASSES = 19
IGNORE_INDEX = 255

CITYSCAPES_SEGMENTATION_CLASSES = [
    (0, 'road'),
    (1, 'sidewalk'),
    (2, 'building'),
    (3, 'wall'),
    (4, 'fence'),
    (5, 'pole'),
    (6, 'traffic light'),
    (7, 'traffic sign'),
    (8, 'vegetation'),
    (9, 'terrain'),
    (10, 'sky'),
    (11, 'person'),
    (12, 'rider'),
    (13, 'car'),
    (14, 'truck'),
    (15, 'bus'),
    (16, 'train'),
    (17, 'motorcycle'),
    (18, 'bicycle'),
]

# ============================================================
# LOAD ALL TRAIN MASKS
# ============================================================

all_mask_files = sorted(
    MASKS_DIR.rglob("*_gtFine_labelTrainIds.png")
)

print(f"Found {len(all_mask_files)} training masks")

assert len(all_mask_files) > 0, "No masks found."

# ============================================================
# PIXEL COUNTING
# ============================================================

pixel_counts = np.zeros(NUM_CLASSES, dtype=np.int64)

total_valid_pixels = 0

for mask_path in tqdm(all_mask_files):

    mask = np.array(Image.open(mask_path), dtype=np.uint8)

    # Remove ignore index pixels
    valid_mask = mask != IGNORE_INDEX
    valid_pixels = mask[valid_mask]

    total_valid_pixels += valid_pixels.size

    # Fast bincount
    counts = np.bincount(
        valid_pixels,
        minlength=NUM_CLASSES
    )

    pixel_counts += counts[:NUM_CLASSES]

# ============================================================
# CLASS FREQUENCIES
# ============================================================

frequencies = pixel_counts / total_valid_pixels

# Avoid divide-by-zero
eps = 1e-12
frequencies = np.clip(frequencies, eps, None)
#print(f"Frequencies= {frequencies})
# ============================================================
# MEDIAN FREQUENCY BALANCING
# ============================================================

median_freq = np.median(frequencies)

class_weights = median_freq / frequencies

# ============================================================
# OPTIONAL NORMALIZATION
# Keeps average weight around 1.0
# ============================================================

class_weights = class_weights / class_weights.mean()

# ============================================================
# PRINT RESULTS
# ============================================================

print("\n============================================================")
print("PIXEL DISTRIBUTION")
print("============================================================")

print(f'{"ID":>3}  {"Class":20s}  {"Pixels":>15}  {"Percent":>10}  {"Weight":>10}')
print('-' * 75)

for class_id, class_name in CITYSCAPES_SEGMENTATION_CLASSES:

    pct = frequencies[class_id] * 100
    weight = class_weights[class_id]

    print(
        f'{class_id:>3}  '
        f'{class_name:20s}  '
        f'{pixel_counts[class_id]:>15,}  '
        f'{pct:>9.4f}%  '
        f'{weight:>10.4f}'
    )

# ============================================================
# PYTHON LIST FOR CONSTANTS FILE
# ============================================================

weights_list = class_weights.tolist()

print("\n============================================================")
print("COPY THIS INTO constants/__init__.py")
print("============================================================\n")

print(f"SEGMENTATION_CLASS_WEIGHTS = {weights_list}")

# ============================================================
# TORCH VERSION
# ============================================================

weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32
)

print("\nTorch tensor:")
print(weights_tensor)

Found 2678 training masks


100%|██████████| 2678/2678 [00:43<00:00, 61.73it/s]


PIXEL DISTRIBUTION
 ID  Class                          Pixels     Percent      Weight
---------------------------------------------------------------------------
  0  road                    1,829,732,169    36.8487%      0.0116
  1  sidewalk                  303,546,876     6.1131%      0.0698
  2  building                1,135,671,387    22.8711%      0.0187
  3  wall                       32,764,461     0.6598%      0.6470
  4  fence                      43,886,966     0.8838%      0.4831
  5  pole                       61,186,310     1.2322%      0.3465
  6  traffic light              10,414,495     0.2097%      2.0356
  7  traffic sign               27,480,273     0.5534%      0.7715
  8  vegetation                789,395,620    15.8975%      0.0269
  9  terrain                    58,235,523     1.1728%      0.3640
 10  sky                       198,144,840     3.9904%      0.1070
 11  person                     62,032,002     1.2493%      0.3418
 12  rider                       